# 02 — Point-in-Time Feature Research

## Purpose

Build and validate leakage-safe features for predicting a player's FPL points in a target fixture.

The target fixture row is the modelling observation. Every predictor must be either:
- known from the fixture schedule/context before kickoff; or
- derived strictly from fixtures with `historical_kickoff_time < target_kickoff_time`.

This notebook researches feature definitions only. Scaling, feature selection, hyperparameter tuning, and model fitting belong in `03_model_experiments.ipynb`.


## Research plan

1. Load the canonical player-fixture artifact from Notebook 01.
2. Define the current fixture's `total_points` as the target — no `shift(-1)`.
3. Build player history with explicit `shift(1)` before rolling/cumulative calculations.
4. Add prior-minutes and per-90 features with minimum-history safeguards.
5. Build one row per team-fixture, then derive leakage-safe team/opponent history.
6. Merge only through validated many-to-one keys.
7. Add pre-kickoff fixture context and cold-start indicators.
8. Test temporal logic with synthetic examples, not only column-name checks.
9. Maintain a feature availability contract before anything is allowed into modelling.
10. Export only after all invariants pass.

**Deliberately deferred:** gameweek-window aggregates. Fixture chronology is the baseline because postponed fixtures and DGWs make plain GW ordering unsafe.


In [1]:
from pathlib import Path
import os

os.environ.pop("MPLBACKEND", None)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 250)

CANONICAL_PATH = Path("./data/processed/fpl_canonical.parquet")

OUTPUT_DIR = Path("./data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_OUTPUT = OUTPUT_DIR / "fixture_training_features.parquet"
FEATURE_DICTIONARY_OUTPUT = OUTPUT_DIR / "feature_dictionary.csv"

print("Canonical input:", CANONICAL_PATH.resolve())
print("Feature output:", FEATURE_OUTPUT.resolve())


Canonical input: /home/l/fpl_prediction/notebooks/data/processed/fpl_canonical.parquet
Feature output: /home/l/fpl_prediction/notebooks/data/processed/fixture_training_features.parquet


In [2]:
df = pd.read_parquet(CANONICAL_PATH)

required = {
    "season",
    "player_id",
    "fixture_id",
    "kickoff_time",
    "total_points",
    "team",
    "opponent_team",
    "opp_team_name",
}
missing_required = required - set(df.columns)
assert not missing_required, (
    f"Canonical artifact missing required columns: {sorted(missing_required)}"
)

df["kickoff_time"] = pd.to_datetime(df["kickoff_time"], utc=True)

# ------------------------------------------------------------
# Explicit team identity schema
#
# Canonical Notebook 01 currently stores:
#   team          -> team name
#   opponent_team -> numeric team ID
#   opp_team_name -> opponent team name
#
# For feature joins, derive symmetric numeric/name columns and
# avoid joining a team name to a numeric team ID.
# ------------------------------------------------------------

df["team_name"] = df["team"]
df["opponent_team_id"] = pd.to_numeric(
    df["opponent_team"],
    errors="coerce",
).astype("Int64")
df["opponent_team_name"] = df["opp_team_name"]

team_id_name_pairs = (
    df[
        ["season", "opponent_team_id", "opponent_team_name"]
    ]
    .dropna()
    .drop_duplicates()
)

id_name_conflicts = (
    team_id_name_pairs
    .groupby(["season", "opponent_team_id"])["opponent_team_name"]
    .nunique()
)

assert (id_name_conflicts == 1).all(), (
    "A season/team ID maps to multiple team names."
)

name_id_pairs = (
    team_id_name_pairs
    .rename(
        columns={
            "opponent_team_id": "team_id",
            "opponent_team_name": "team_name",
        }
    )
)

name_id_conflicts = (
    name_id_pairs
    .assign(
        _team_name_key=lambda x: (
            x["team_name"].astype(str).str.strip().str.casefold()
        )
    )
    .groupby(["season", "_team_name_key"])["team_id"]
    .nunique()
)

assert (name_id_conflicts == 1).all(), (
    "A season/team name maps to multiple team IDs."
)

team_name_to_id = (
    name_id_pairs
    .assign(
        _team_name_key=lambda x: (
            x["team_name"].astype(str).str.strip().str.casefold()
        )
    )
    .drop_duplicates(["season", "_team_name_key"])
    .set_index(["season", "_team_name_key"])["team_id"]
    .to_dict()
)

team_name_key = (
    df["team_name"].astype(str).str.strip().str.casefold()
)

df["team_id"] = [
    team_name_to_id.get((season, name_key), pd.NA)
    for season, name_key in zip(df["season"], team_name_key)
]
df["team_id"] = pd.array(df["team_id"], dtype="Int64")

assert df["team_id"].notna().all(), (
    "Some team names could not be mapped to season-specific team IDs."
)
assert df["opponent_team_id"].notna().all(), (
    "Some opponent team IDs are missing."
)
assert not (df["team_id"] == df["opponent_team_id"]).any(), (
    "Found rows where team_id equals opponent_team_id."
)

sort_cols = [
    c for c in ["season", "kickoff_time", "fixture_id", "player_id"]
    if c in df.columns
]
df = df.sort_values(sort_cols, kind="stable").reset_index(drop=True)

assert not df.duplicated(["season", "fixture_id", "player_id"]).any()

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("\nTeam identity dtypes:")
print(
    df[
        [
            "team_id",
            "team_name",
            "opponent_team_id",
            "opponent_team_name",
        ]
    ].dtypes
)
display(
    df[
        [
            "season",
            "fixture_id",
            "team_id",
            "team_name",
            "opponent_team_id",
            "opponent_team_name",
        ]
    ].head()
)


Rows: 96,169
Columns: 43

Team identity dtypes:
team_id               Int64
team_name               str
opponent_team_id      Int64
opponent_team_name      str
dtype: object


,season,fixture_id,team_id,team_name,opponent_team_id,opponent_team_name
0,2016-17,4,7,Hull,8,Leicester
1,2016-17,4,7,Hull,8,Leicester
2,2016-17,4,7,Hull,8,Leicester
3,2016-17,4,7,Hull,8,Leicester
4,2016-17,4,7,Hull,8,Leicester


## 1. Define the target directly from actual fixtures

Each canonical row already represents a real player-fixture observation, so the target is the current row's `total_points`.

Do **not** use `shift(-1)`. That approach can break under postponed fixtures, DGWs, and imperfect row ordering.

Zero-minute rows are retained deliberately. The baseline task is therefore a **single-stage expected-points problem**: the model must learn both playing-time likelihood and points conditional on playing. A two-stage `P(minutes) × E[points | plays]` formulation can be compared later.


In [3]:
target_required = [
    c for c in [
        "season",
        "player_id",
        "player_name",
        "fixture_id",
        "gameweek",
        "kickoff_time",
        "team_id",
        "team_name",
        "opponent_team_id",
        "opponent_team_name",
        "was_home",
        "position",
        "value",
        "total_points",
    ]
    if c in df.columns
]

targets = df[target_required].copy()
targets = targets.rename(columns={"total_points": "target_points"})

print("Target rows:", len(targets))
display(targets.head())


Target rows: 96169


,season,player_id,player_name,fixture_id,gameweek,kickoff_time,team_id,team_name,opponent_team_id,opponent_team_name,was_home,position,value,target_points
0,2016-17,147,Eldin Jakupovic,4,1,2016-08-13 11:30:00+00:00,7,Hull,8,Leicester,True,GK,40,3
1,2016-17,152,Andrew Robertson,4,1,2016-08-13 11:30:00+00:00,7,Hull,8,Leicester,True,DEF,45,2
2,2016-17,153,Harry Maguire,4,1,2016-08-13 11:30:00+00:00,7,Hull,8,Leicester,True,DEF,45,0
3,2016-17,155,Robert Snodgrass,4,1,2016-08-13 11:30:00+00:00,7,Hull,8,Leicester,True,MID,55,10
4,2016-17,157,Jake Livermore,4,1,2016-08-13 11:30:00+00:00,7,Hull,8,Leicester,True,MID,50,2


## 2. Define historical-stat columns

These columns are outcomes of past fixtures.

They may contribute to features only through records where:

`historical.kickoff_time < target.kickoff_time`

In [4]:
PLAYER_HISTORY_STATS = [
    c for c in [
        "minutes",
        "total_points",
        "goals_scored",
        "assists",
        "bonus",
        "bps",
        "clean_sheets",
        "goals_conceded",
        "saves",
        "yellow_cards",
        "red_cards",
        "own_goals",
        "penalties_saved",
        "penalties_missed",
        "creativity",
        "influence",
        "threat",
        "ict_index",
    ]
    if c in df.columns
]

PLAYER_HISTORY_STATS

['minutes',
 'total_points',
 'goals_scored',
 'assists',
 'bonus',
 'bps',
 'clean_sheets',
 'goals_conceded',
 'saves',
 'yellow_cards',
 'red_cards',
 'own_goals',
 'penalties_saved',
 'penalties_missed',
 'creativity',
 'influence',
 'threat',
 'ict_index']

## 3. Player-history feature builder

For correctness and clarity, the first implementation is explicit rather than maximally optimized.

Once feature definitions are stable, the same semantics can be rewritten using vectorized operations or a feature store.

Important:
- sort by kickoff time, not just GW;
- call `.shift(1)` before expanding/rolling so the target fixture never contributes to its own features.

In [5]:
def add_player_history_features(
    data: pd.DataFrame,
    windows=(3, 6),
    stats=None,
) -> pd.DataFrame:
    # Every target-row feature is based on rows strictly before that fixture.
    stats = stats or PLAYER_HISTORY_STATS

    out = data.copy()
    out = out.sort_values(
        ["season", "player_id", "kickoff_time", "fixture_id"],
        kind="stable",
    ).reset_index(drop=True)

    group_keys = ["season", "player_id"]
    grouped = out.groupby(group_keys, sort=False)

    out["player_prior_fixtures"] = grouped.cumcount()

    if "minutes" in out.columns:
        prior_minutes = grouped["minutes"].shift(1)

        played_60plus = (out["minutes"] >= 60).astype(float)
        prior_60plus = played_60plus.groupby(
            [out[k] for k in group_keys],
            sort=False,
        ).shift(1)

        for window in windows:
            keys = [out[k] for k in group_keys]

            out[f"minutes_last_{window}_fixtures"] = (
                prior_minutes.groupby(keys, sort=False)
                .rolling(window=window, min_periods=1)
                .sum()
                .reset_index(level=[0, 1], drop=True)
            )

            out[f"avg_minutes_last_{window}_fixtures"] = (
                prior_minutes.groupby(keys, sort=False)
                .rolling(window=window, min_periods=1)
                .mean()
                .reset_index(level=[0, 1], drop=True)
            )

            out[f"appearances_60plus_last_{window}_fixtures"] = (
                prior_60plus.groupby(keys, sort=False)
                .rolling(window=window, min_periods=1)
                .sum()
                .reset_index(level=[0, 1], drop=True)
            )

    for stat in stats:
        prior = grouped[stat].shift(1)
        keys = [out[k] for k in group_keys]

        out[f"{stat}_season_to_date"] = (
            prior.fillna(0)
            .groupby(keys, sort=False)
            .cumsum()
        )

        for window in windows:
            out[f"{stat}_last_{window}_fixtures"] = (
                prior.groupby(keys, sort=False)
                .rolling(window=window, min_periods=1)
                .sum()
                .reset_index(level=[0, 1], drop=True)
            )

            out[f"{stat}_avg_last_{window}_fixtures"] = (
                prior.groupby(keys, sort=False)
                .rolling(window=window, min_periods=1)
                .mean()
                .reset_index(level=[0, 1], drop=True)
            )

    return out


In [6]:
features = add_player_history_features(df, windows=(3, 6))
print(features.shape)
display(features.head())

(96169, 138)


,season,player_name,position,team,assists,bonus,bps,clean_sheets,creativity,player_id,fixture_id,goals_conceded,goals_scored,ict_index,influence,kickoff_time,minutes,opponent_team,opp_team_name,own_goals,penalties_missed,penalties_saved,red_cards,round,saves,selected,team_a_score,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,gameweek,team_recovery_method,position_recovery_method,team_name,opponent_team_id,opponent_team_name,team_id,player_prior_fixtures,minutes_last_3_fixtures,avg_minutes_last_3_fixtures,appearances_60plus_last_3_fixtures,minutes_last_6_fixtures,avg_minutes_last_6_fixtures,appearances_60plus_last_6_fixtures,minutes_season_to_date,minutes_avg_last_3_fixtures,minutes_avg_last_6_fixtures,total_points_season_to_date,total_points_last_3_fixtures,total_points_avg_last_3_fixtures,total_points_last_6_fixtures,total_points_avg_last_6_fixtures,goals_scored_season_to_date,goals_scored_last_3_fixtures,goals_scored_avg_last_3_fixtures,goals_scored_last_6_fixtures,goals_scored_avg_last_6_fixtures,assists_season_to_date,assists_last_3_fixtures,assists_avg_last_3_fixtures,assists_last_6_fixtures,assists_avg_last_6_fixtures,bonus_season_to_date,bonus_last_3_fixtures,bonus_avg_last_3_fixtures,bonus_last_6_fixtures,bonus_avg_last_6_fixtures,bps_season_to_date,bps_last_3_fixtures,bps_avg_last_3_fixtures,bps_last_6_fixtures,bps_avg_last_6_fixtures,clean_sheets_season_to_date,clean_sheets_last_3_fixtures,clean_sheets_avg_last_3_fixtures,clean_sheets_last_6_fixtures,clean_sheets_avg_last_6_fixtures,goals_conceded_season_to_date,goals_conceded_last_3_fixtures,goals_conceded_avg_last_3_fixtures,goals_conceded_last_6_fixtures,goals_conceded_avg_last_6_fixtures,saves_season_to_date,saves_last_3_fixtures,saves_avg_last_3_fixtures,saves_last_6_fixtures,saves_avg_last_6_fixtures,yellow_cards_season_to_date,yellow_cards_last_3_fixtures,yellow_cards_avg_last_3_fixtures,yellow_cards_last_6_fixtures,yellow_cards_avg_last_6_fixtures,red_cards_season_to_date,red_cards_last_3_fixtures,red_cards_avg_last_3_fixtures,red_cards_last_6_fixtures,red_cards_avg_last_6_fixtures,own_goals_season_to_date,own_goals_last_3_fixtures,own_goals_avg_last_3_fixtures,own_goals_last_6_fixtures,own_goals_avg_last_6_fixtures,penalties_saved_season_to_date,penalties_saved_last_3_fixtures,penalties_saved_avg_last_3_fixtures,penalties_saved_last_6_fixtures,penalties_saved_avg_last_6_fixtures,penalties_missed_season_to_date,penalties_missed_last_3_fixtures,penalties_missed_avg_last_3_fixtures,penalties_missed_last_6_fixtures,penalties_missed_avg_last_6_fixtures,creativity_season_to_date,creativity_last_3_fixtures,creativity_avg_last_3_fixtures,creativity_last_6_fixtures,creativity_avg_last_6_fixtures,influence_season_to_date,influence_last_3_fixtures,influence_avg_last_3_fixtures,influence_last_6_fixtures,influence_avg_last_6_fixtures,threat_season_to_date,threat_last_3_fixtures,threat_avg_last_3_fixtures,threat_last_6_fixtures,threat_avg_last_6_fixtures,ict_index_season_to_date,ict_index_last_3_fixtures,ict_index_avg_last_3_fixtures,ict_index_last_6_fixtures,ict_index_avg_last_6_fixtures
0,2016-17,Héctor Bellerín,DEF,Arsenal,0,0,9,0,6.8,6,8,4,0,2.8,19.6,2016-08-14 15:00:00+00:00,90,9,Liverpool,0,0,0,0,1,0,950852,4.0,3.0,2.0,0,0,0,0,65,True,0,1,fixture_opponent_inference,observed_normalized,Arsenal,9,Liverpool,1,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
1,2016-17,Héctor Bellerín,DEF,Arsenal,0,3,31,1,28.9,6,13,0,0,6.1,20.8,2016-08-20 16:30:00+00:00,90,8,Leicester,0,0,0,0,2,0,970078,0.0,0.0,11.0,9,-56304,4952,61256,65,False,0,2,fixture_opponent_inference,observed_normalized,Arsenal,8,L

### Leakage tests for player-history features

Two checks are used:
1. real-data cold-start rows must have zero prior fixtures;
2. a tiny synthetic sequence must reproduce known rolling values exactly.

The synthetic test catches implementation leakage that a simple column-name check cannot.


In [7]:
first_rows = (
    features.sort_values(["season", "player_id", "kickoff_time", "fixture_id"])
    .groupby(["season", "player_id"], as_index=False)
    .head(1)
)
assert (first_rows["player_prior_fixtures"] == 0).all()

synthetic_player = pd.DataFrame({
    "season": ["S"] * 3,
    "player_id": [1] * 3,
    "fixture_id": [101, 102, 103],
    "kickoff_time": pd.to_datetime(
        ["2025-08-01", "2025-08-08", "2025-08-15"], utc=True
    ),
    "minutes": [90, 90, 90],
    "total_points": [2, 5, 10],
})

synthetic_features = add_player_history_features(
    synthetic_player,
    windows=(3,),
    stats=["total_points"],
)

actual = synthetic_features["total_points_last_3_fixtures"].tolist()
assert pd.isna(actual[0])
assert actual[1:] == [2.0, 7.0], (
    f"Player rolling-history leakage test failed: {actual}"
)

print("Player-history leakage tests passed.")


Player-history leakage tests passed.


## 4. Prior per-90 features

Per-90 rates use **season-to-date history excluding the target fixture**.

Rates are left missing until the player has accumulated enough prior minutes. This avoids unstable values from tiny samples; the accompanying low-history flag preserves the reason for missingness.

The initial threshold is 90 prior minutes and should later be tested inside chronological training folds.


In [8]:
PER90_STATS = [
    c for c in [
        "total_points", "goals_scored", "assists", "bonus", "bps",
        "clean_sheets", "goals_conceded", "saves",
        "creativity", "influence", "threat", "ict_index",
    ]
    if c in df.columns
]

MIN_PRIOR_MINUTES_FOR_RATE = 90

if "minutes_season_to_date" in features.columns:
    prior_minutes = features["minutes_season_to_date"]

    for stat in PER90_STATS:
        cumulative_col = f"{stat}_season_to_date"
        if cumulative_col not in features.columns:
            continue

        rate = features[cumulative_col] / prior_minutes.replace(0, np.nan) * 90
        rate = rate.where(prior_minutes >= MIN_PRIOR_MINUTES_FOR_RATE)

        features[f"{stat}_per90_season_to_date"] = rate

    features["low_history_minutes_flag"] = (
        prior_minutes < MIN_PRIOR_MINUTES_FOR_RATE
    ).astype(int)

/tmp/ipykernel_47853/4092169907.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"{stat}_per90_season_to_date"] = rate
/tmp/ipykernel_47853/4092169907.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"{stat}_per90_season_to_date"] = rate
/tmp/ipykernel_47853/4092169907.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-

## 5. Gameweek-window features — deferred

Do not create rolling features by plain `gameweek` order in the baseline table.

A postponed GW14 fixture can occur after GW15, and two fixtures in the same DGW can occur on different dates. Fixture-level windows ordered by `kickoff_time` are therefore the canonical baseline.

A future gameweek aggregate must use an **as-of target kickoff** rule and include only fully completed fixtures/gameweeks that were known at that timestamp.


In [9]:
GAMEWEEK_WINDOW_FEATURES_ENABLED = False
print("Gameweek-window features enabled:", GAMEWEEK_WINDOW_FEATURES_ENABLED)


Gameweek-window features enabled: False


**Decision:** use last-3 / last-6 **fixture** windows for the initial experiments. Revisit GW windows only after implementing explicit target-time as-of joins.


## 6. Build team-match history safely

The reference notebook creates `teams_df`, which is a good idea, but later suffers a many-to-many merge.

Here we first collapse player rows to **one team-fixture row**, validate uniqueness, then engineer prior team features.

A team-feature table must be unique on:

`season + fixture_id + team`

or an equally strong fixture key.

In [10]:
team_required = [
    c for c in [
        "season",
        "fixture_id",
        "gameweek",
        "kickoff_time",
        "team_id",
        "team_name",
        "opponent_team_id",
        "opponent_team_name",
        "was_home",
    ]
    if c in df.columns
]

team_source_cols = team_required.copy()

for c in ["team_h_score", "team_a_score"]:
    if c in df.columns:
        team_source_cols.append(c)

# One row per participating team per actual fixture.
team_matches = (
    df[team_source_cols]
    .drop_duplicates()
    .copy()
)

team_key = ["season", "fixture_id", "team_id"]

missing_team_key = [
    c for c in team_key
    if c not in team_matches.columns
]

assert not missing_team_key, (
    f"Missing team-fixture key columns: {missing_team_key}"
)

duplicated = team_matches.duplicated(team_key, keep=False)

print("Duplicate team-fixture rows:", int(duplicated.sum()))

if duplicated.any():
    display(
        team_matches.loc[duplicated]
        .sort_values(team_key)
        .head(100)
    )

assert not duplicated.any(), (
    "Team fixture table is not unique on "
    "(season, fixture_id, team_id)."
)

print("Team fixtures:", f"{len(team_matches):,}")
display(team_matches.head())


Duplicate team-fixture rows: 0
Team fixtures: 3,800


,season,fixture_id,gameweek,kickoff_time,team_id,team_name,opponent_team_id,opponent_team_name,was_home,team_h_score,team_a_score
0,2016-17,4,1,2016-08-13 11:30:00+00:00,7,Hull,8,Leicester,True,2.0,1.0
5,2016-17,4,1,2016-08-13 11:30:00+00:00,8,Leicester,7,Hull,False,2.0,1.0
17,2016-17,1,1,2016-08-13 14:00:00+00:00,3,Burnley,16,Swansea,True,0.0,1.0
24,2016-17,1,1,2016-08-13 14:00:00+00:00,16,Swansea,3,Burnley,False,0.0,1.0
31,2016-17,2,1,2016-08-13 14:00:00+00:00,5,Crystal Palace,19,West Brom,True,0.0,1.0


In [11]:
def add_team_score_columns(team_matches):
    out = team_matches.copy()

    if not {"team_h_score", "team_a_score", "was_home"}.issubset(out.columns):
        print("Team score columns unavailable; skipping score derivation.")
        return out

    out["goals_for"] = np.where(
        out["was_home"].astype(bool),
        out["team_h_score"],
        out["team_a_score"],
    )
    out["goals_against"] = np.where(
        out["was_home"].astype(bool),
        out["team_a_score"],
        out["team_h_score"],
    )

    out["win"] = (out["goals_for"] > out["goals_against"]).astype(int)
    out["draw"] = (out["goals_for"] == out["goals_against"]).astype(int)
    out["loss"] = (out["goals_for"] < out["goals_against"]).astype(int)

    return out

team_matches = add_team_score_columns(team_matches)

## 7. Team attacking form before the target fixture

All rolling team features use `.shift(1)`.

That means Arsenal-vs-Chelsea cannot use Arsenal's goals from Arsenal-vs-Chelsea itself.

In [12]:
def add_team_history_features(team_matches, windows=(3, 6)):
    out = team_matches.sort_values(
        ["season", "team_id", "kickoff_time", "fixture_id"],
        kind="stable",
    ).copy()

    group_keys = ["season", "team_id"]

    available = [
        c for c in [
            "goals_for",
            "goals_against",
            "win",
            "draw",
            "loss",
        ]
        if c in out.columns
    ]

    grouped = out.groupby(group_keys, sort=False)
    out["team_prior_fixtures"] = grouped.cumcount()

    for stat in available:
        prior = grouped[stat].shift(1)
        keys = [out[k] for k in group_keys]

        out[f"team_{stat}_season_to_date"] = (
            prior.fillna(0)
            .groupby(keys, sort=False)
            .cumsum()
        )

        for window in windows:
            out[f"team_{stat}_avg_last_{window}"] = (
                prior.groupby(keys, sort=False)
                .rolling(
                    window=window,
                    min_periods=1,
                )
                .mean()
                .reset_index(level=[0, 1], drop=True)
            )

    return out


team_features = add_team_history_features(team_matches)

display(
    team_features[
        [
            c for c in [
                "season",
                "fixture_id",
                "team_id",
                "team_name",
                "opponent_team_id",
                "opponent_team_name",
                "team_prior_fixtures",
                "team_goals_for_avg_last_3",
            ]
            if c in team_features.columns
        ]
    ].head()
)


,season,fixture_id,team_id,team_name,opponent_team_id,opponent_team_name,team_prior_fixtures,team_goals_for_avg_last_3
141,2016-17,8,1,Arsenal,9,Liverpool,0,NaN
338,2016-17,13,1,Arsenal,8,Leicester,1,3.000000
517,2016-17,28,1,Arsenal,18,Watford,2,1.500000
622,2016-17,31,1,Arsenal,13,Southampton,3,2.000000
835,2016-17,43,1,Arsenal,7,Hull,4,1.666667


### Leakage test for team history

A synthetic three-fixture sequence verifies that the current fixture score never enters its own team-history features.


In [13]:
synthetic_team = pd.DataFrame({
    "season": ["S"] * 3,
    "team_id": [1] * 3,
    "fixture_id": [1, 2, 3],
    "kickoff_time": pd.to_datetime(
        ["2025-08-01", "2025-08-08", "2025-08-15"],
        utc=True,
    ),
    "goals_for": [1, 2, 4],
    "goals_against": [0, 1, 3],
    "win": [1, 1, 1],
    "draw": [0, 0, 0],
    "loss": [0, 0, 0],
})

synthetic_team_features = add_team_history_features(
    synthetic_team,
    windows=(3,),
)

actual = synthetic_team_features[
    "team_goals_for_avg_last_3"
].tolist()

assert pd.isna(actual[0])
assert actual[1] == 1.0
assert actual[2] == 1.5, (
    f"Team rolling-history leakage test failed: {actual}"
)

print("Team-history leakage test passed.")


Team-history leakage test passed.


## 8. Opponent defensive strength

Rather than encoding the opponent name and hoping the model learns identity, expose meaningful opponent history.

For a target fixture:
- target player's team history becomes `team_*`;
- opponent's team history becomes `opp_*`.

This is one of the largest conceptual improvements over the reference notebook's final feature set.

In [14]:
required_opponent_cols = {
    "season",
    "fixture_id",
    "team_id",
    "opponent_team_id",
}

if required_opponent_cols.issubset(team_features.columns):
    generated_team_history_cols = [
        c for c in team_features.columns
        if c == "team_prior_fixtures"
        or c.startswith("team_goals_")
        or c.startswith("team_win_")
        or c.startswith("team_draw_")
        or c.startswith("team_loss_")
    ]

    # team_features contains the history of the team represented
    # by team_id. To attach that history as the current player's
    # opponent history, swap the two numeric team identifiers.
    opponent_features = team_features[
        [
            "season",
            "fixture_id",
            "team_id",
            "opponent_team_id",
        ]
        + generated_team_history_cols
    ].copy()

    opponent_features = opponent_features.rename(
        columns={
            "team_id": "opponent_team_id",
            "opponent_team_id": "team_id",
            **{
                c: c.replace("team_", "opp_", 1)
                for c in generated_team_history_cols
            },
        }
    )

    opponent_key = [
        "season",
        "fixture_id",
        "team_id",
        "opponent_team_id",
    ]

    assert not opponent_features.duplicated(
        opponent_key
    ).any(), (
        "Opponent feature table is not unique on its fixture/team key."
    )

    print("Opponent feature key dtypes:")
    print(opponent_features[opponent_key].dtypes)

    display(opponent_features.head())
else:
    opponent_features = None
    print(
        "Opponent feature construction requires "
        "season, fixture_id, team_id and opponent_team_id."
    )


Opponent feature key dtypes:
season                str
fixture_id          Int64
team_id             Int64
opponent_team_id    Int64
dtype: object


,season,fixture_id,opponent_team_id,team_id,opp_prior_fixtures,opp_goals_for_season_to_date,opp_goals_for_avg_last_3,opp_goals_for_avg_last_6,opp_goals_against_season_to_date,opp_goals_against_avg_last_3,opp_goals_against_avg_last_6,opp_win_season_to_date,opp_win_avg_last_3,opp_win_avg_last_6,opp_draw_season_to_date,opp_draw_avg_last_3,opp_draw_avg_last_6,opp_loss_season_to_date,opp_loss_avg_last_3,opp_loss_avg_last_6
141,2016-17,8,1,9,0,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN
338,2016-17,13,1,8,1,3.0,3.000000,3.0,4.0,4.000000,4.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,1.0,1.000000,1.000000
517,2016-17,28,1,18,2,3.0,1.500000,1.5,4.0,2.000000,2.000000,0.0,0.000000,0.000000,1.0,0.500000,0.500000,1.0,0.500000,0.500000
622,2016-17,31,1,13,3,6.0,2.000000,2.0,5.0,1.666667,1.666667,1.0,0.333333,0.333333,1.0,0.333333,0.333333,1.0,0.333333,0.333333
835,2016-17,43,1,7,4,8.0,1.666667,2.0,6.0,0.666667,1.500000,2.0,0.666667,0.500000,1.0,0.333333,0.250000,1.0,0.000000,0.250000


## 9. Join invariants — prevent the reference notebook's row explosion

Before every merge:

1. assert the feature side is unique on its join key;
2. record left row count;
3. merge with `validate=...`;
4. assert output row count is unchanged.

Never accept an unexplained 93k → 158k expansion.

In [15]:
def safe_left_merge(
    left,
    right,
    on,
    relationship="many_to_one",
    label="merge",
):
    before = len(left)

    missing_left = [c for c in on if c not in left.columns]
    missing_right = [c for c in on if c not in right.columns]

    if missing_left or missing_right:
        raise KeyError(
            f"{label}: missing join keys. "
            f"left={missing_left}, right={missing_right}"
        )

    dtype_mismatches = {}

    for key in on:
        left_dtype = left[key].dtype
        right_dtype = right[key].dtype

        if left_dtype != right_dtype:
            dtype_mismatches[key] = {
                "left_dtype": str(left_dtype),
                "right_dtype": str(right_dtype),
                "left_examples": (
                    left[key].dropna().head(3).tolist()
                ),
                "right_examples": (
                    right[key].dropna().head(3).tolist()
                ),
            }

    if dtype_mismatches:
        raise TypeError(
            f"{label}: incompatible join-key schemas: "
            f"{dtype_mismatches}"
        )

    merged = left.merge(
        right,
        how="left",
        on=on,
        validate=relationship,
    )

    after = len(merged)

    if after != before:
        raise AssertionError(
            f"{label}: row count changed from "
            f"{before:,} to {after:,}"
        )

    return merged


## 10. Assemble fixture-level research table

The target fixture row already contains pre-kickoff context such as:
- team;
- opponent;
- home/away;
- position;
- price/value if the source records its pre-deadline value.

Historical aggregates are attached from the same row because they were computed with `.shift(1)`.

Do **not** include raw target-fixture outcomes such as current-row `minutes`, `total_points`, goals or BPS as model inputs.

In [16]:
identity_cols = [
    c for c in [
        "season",
        "player_id",
        "player_name",
        "fixture_id",
        "gameweek",
        "kickoff_time",
        "team_id",
        "team_name",
        "opponent_team_id",
        "opponent_team_name",
        "position",
        "was_home",
        "value",
    ]
    if c in features.columns
]

history_feature_cols = [
    c for c in features.columns
    if (
        "_last_" in c
        or c.endswith("_season_to_date")
        or "_per90_" in c
        or c in [
            "player_prior_fixtures",
            "low_history_minutes_flag",
        ]
    )
]

research = features[
    identity_cols
    + history_feature_cols
    + ["total_points"]
].copy()

research = research.rename(
    columns={"total_points": "target_points"}
)

print(research.shape)
display(research.head())


(96169, 122)


,season,player_id,player_name,fixture_id,gameweek,kickoff_time,team_id,team_name,opponent_team_id,opponent_team_name,position,was_home,value,player_prior_fixtures,minutes_last_3_fixtures,avg_minutes_last_3_fixtures,appearances_60plus_last_3_fixtures,minutes_last_6_fixtures,avg_minutes_last_6_fixtures,appearances_60plus_last_6_fixtures,minutes_season_to_date,minutes_avg_last_3_fixtures,minutes_avg_last_6_fixtures,total_points_season_to_date,total_points_last_3_fixtures,total_points_avg_last_3_fixtures,total_points_last_6_fixtures,total_points_avg_last_6_fixtures,goals_scored_season_to_date,goals_scored_last_3_fixtures,goals_scored_avg_last_3_fixtures,goals_scored_last_6_fixtures,goals_scored_avg_last_6_fixtures,assists_season_to_date,assists_last_3_fixtures,assists_avg_last_3_fixtures,assists_last_6_fixtures,assists_avg_last_6_fixtures,bonus_season_to_date,bonus_last_3_fixtures,bonus_avg_last_3_fixtures,bonus_last_6_fixtures,bonus_avg_last_6_fixtures,bps_season_to_date,bps_last_3_fixtures,bps_avg_last_3_fixtures,bps_last_6_fixtures,bps_avg_last_6_fixtures,clean_sheets_season_to_date,clean_sheets_last_3_fixtures,clean_sheets_avg_last_3_fixtures,clean_sheets_last_6_fixtures,clean_sheets_avg_last_6_fixtures,goals_conceded_season_to_date,goals_conceded_last_3_fixtures,goals_conceded_avg_last_3_fixtures,goals_conceded_last_6_fixtures,goals_conceded_avg_last_6_fixtures,saves_season_to_date,saves_last_3_fixtures,saves_avg_last_3_fixtures,saves_last_6_fixtures,saves_avg_last_6_fixtures,yellow_cards_season_to_date,yellow_cards_last_3_fixtures,yellow_cards_avg_last_3_fixtures,yellow_cards_last_6_fixtures,yellow_cards_avg_last_6_fixtures,red_cards_season_to_date,red_cards_last_3_fixtures,red_cards_avg_last_3_fixtures,red_cards_last_6_fixtures,red_cards_avg_last_6_fixtures,own_goals_season_to_date,own_goals_last_3_fixtures,own_goals_avg_last_3_fixtures,own_goals_last_6_fixtures,own_goals_avg_last_6_fixtures,penalties_saved_season_to_date,penalties_saved_last_3_fixtures,penalties_saved_avg_last_3_fixtures,penalties_saved_last_6_fixtures,penalties_saved_avg_last_6_fixtures,penalties_missed_season_to_date,penalties_missed_last_3_fixtures,penalties_missed_avg_last_3_fixtures,penalties_missed_last_6_fixtures,penalties_missed_avg_last_6_fixtures,creativity_season_to_date,creativity_last_3_fixtures,creativity_avg_last_3_fixtures,creativity_last_6_fixtures,creativity_avg_last_6_fixtures,influence_season_to_date,influence_last_3_fixtures,influence_avg_last_3_fixtures,influence_last_6_fixtures,influence_avg_last_6_fixtures,threat_season_to_date,threat_last_3_fixtures,threat_avg_last_3_fixtures,threat_last_6_fixtures,threat_avg_last_6_fixtures,ict_index_season_to_date,ict_index_last_3_fixtures,ict_index_avg_last_3_fixtures,ict_index_last_6_fixtures,ict_index_avg_last_6_fixtures,total_points_per90_season_to_date,goals_scored_per90_season_to_date,assists_per90_season_to_date,bonus_per90_season_to_date,bps_per90_season_to_date,clean_sheets_per90_season_to_date,goals_conceded_per90_season_to_date,saves_per90_season_to_date,creativity_per90_season_to_date,influence_per90_season_to_date,threat_per90_season_to_date,ict_index_per90_season_to_date,low_history_minutes_flag,target_points
0,2016-17,6,Héctor Bellerín,8,1,2016-08-14 15:00:00+00:00,1,Arsenal,9,Liverpool,DEF,True,65,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
1,2016-17,6,Héctor Bellerín,13,2,2016-08-20 16:30:00+00:00,1,Arsenal,8,Leicester,DEF,False,65,1,90.0,90.0,1.0,90.0,90.0,1.0,90.0,90.0,90.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,9.0,9.0,9.000000,9.0,9.000000,0.0,0.

## 11. Add team/opponent features only through validated keys

Use fixture identifiers, not combinations such as:

`current kickoff_time + next team`

which caused the reference notebook's many-to-many merge.

In [17]:
# ------------------------------------------------------------
# Team history merge
# ------------------------------------------------------------

team_join_keys = [
    "season",
    "fixture_id",
    "team_id",
]

if set(team_join_keys).issubset(research.columns):
    team_feature_cols = [
        c for c in team_features.columns
        if c in team_join_keys
        or c.startswith("team_")
    ]

    team_lookup = team_features[
        team_feature_cols
    ].copy()

    assert not team_lookup.duplicated(
        team_join_keys
    ).any()

    research = safe_left_merge(
        research,
        team_lookup,
        on=team_join_keys,
        relationship="many_to_one",
        label="team feature merge",
    )

# ------------------------------------------------------------
# Opponent history merge
# ------------------------------------------------------------

opponent_join_keys = [
    "season",
    "fixture_id",
    "team_id",
    "opponent_team_id",
]

if (
    opponent_features is not None
    and set(opponent_join_keys).issubset(research.columns)
):
    opp_feature_cols = [
        c for c in opponent_features.columns
        if c in opponent_join_keys
        or c.startswith("opp_")
    ]

    opp_lookup = opponent_features[
        opp_feature_cols
    ].copy()

    assert not opp_lookup.duplicated(
        opponent_join_keys
    ).any()

    # Explicit schema check: both team identifiers must be
    # numeric on both sides. Never cast names to strings merely
    # to make a merge run.
    for key in ["team_id", "opponent_team_id"]:
        assert pd.api.types.is_integer_dtype(
            research[key].dtype
        ), f"{key} is not integer-like in research"

        assert pd.api.types.is_integer_dtype(
            opp_lookup[key].dtype
        ), f"{key} is not integer-like in opponent lookup"

    research = safe_left_merge(
        research,
        opp_lookup,
        on=opponent_join_keys,
        relationship="many_to_one",
        label="opponent feature merge",
    )

print(research.shape)

# ------------------------------------------------------------
# Merge coverage diagnostics
# ------------------------------------------------------------

team_feature_candidates = [
    c for c in research.columns
    if c.startswith("team_")
    and c not in {
        "team_id",
        "team_name",
    }
]

if team_feature_candidates:
    team_probe = team_feature_candidates[0]
    team_match_rate = research[team_probe].notna().mean()
    print(
        f"Team feature match rate ({team_probe}): "
        f"{team_match_rate:.2%}"
    )

opp_feature_candidates = [
    c for c in research.columns
    if c.startswith("opp_")
]

if opp_feature_candidates:
    opp_probe = opp_feature_candidates[0]
    opp_match_rate = research[opp_probe].notna().mean()
    print(
        f"Opponent feature match rate ({opp_probe}): "
        f"{opp_match_rate:.2%}"
    )

# These are diagnostics rather than hard 100% assertions because
# cold-start historical rows can legitimately have missing lag features.


(96169, 157)
Team feature match rate (team_name_x): 100.00%
Opponent feature match rate (opp_prior_fixtures): 100.00%


## 12. Rest and schedule features

Fixture context can add predictive signal while remaining fully point-in-time.

Keep **player rest** and **team schedule rest** separate:

- `days_since_player_fixture` — days since the player's previous recorded fixture;
- `days_since_last_appearance` — days since the player's previous fixture with `minutes > 0`;
- `days_since_team_fixture` — days since the team's previous actual fixture;
- `is_home` — current fixture home/away;
- `fixture_number_in_gameweek` — first, second, etc. team fixture in that gameweek;
- `is_additional_gw_fixture` — whether this is the team's second-or-later fixture in the gameweek;
- `short_rest_3d` / `short_rest_4d` — flags derived from team rest.

Historical rest features use only earlier fixtures. Cold-start missing values are retained rather than replaced with zero.

Future-looking schedule features such as `days_until_next_fixture` are deliberately excluded unless historical point-in-time fixture snapshots become available, because postponed/rescheduled matches can otherwise leak hindsight.

Expected-minutes, injury, suspension and availability features should be added later only from trustworthy pre-deadline sources.


In [18]:
# ============================================================
# 12A. Player rest features
# ============================================================

player_schedule = (
    features[
        [
            "season",
            "player_id",
            "fixture_id",
            "kickoff_time",
            "minutes",
        ]
    ]
    .sort_values(
        [
            "season",
            "player_id",
            "kickoff_time",
            "fixture_id",
        ],
        kind="stable",
    )
    .copy()
)

player_key = [
    "season",
    "fixture_id",
    "player_id",
]

assert not player_schedule.duplicated(
    player_key
).any()

player_schedule["previous_player_fixture_time"] = (
    player_schedule
    .groupby(
        ["season", "player_id"],
        sort=False,
    )["kickoff_time"]
    .shift(1)
)

player_schedule["days_since_player_fixture"] = (
    (
        player_schedule["kickoff_time"]
        - player_schedule["previous_player_fixture_time"]
    )
    .dt.total_seconds()
    .div(86_400)
)

# Current-fixture minutes must not determine the current feature.
# Mark appearance times, carry them forward within a player,
# then shift by one fixture.
player_schedule["_appearance_time"] = (
    player_schedule["kickoff_time"]
    .where(player_schedule["minutes"] > 0)
)

player_schedule["previous_appearance_time"] = (
    player_schedule
    .groupby(
        ["season", "player_id"],
        sort=False,
    )["_appearance_time"]
    .transform(lambda s: s.ffill().shift(1))
)

player_schedule["days_since_last_appearance"] = (
    (
        player_schedule["kickoff_time"]
        - player_schedule["previous_appearance_time"]
    )
    .dt.total_seconds()
    .div(86_400)
)

player_rest_lookup = player_schedule[
    player_key
    + [
        "days_since_player_fixture",
        "days_since_last_appearance",
    ]
].copy()

assert not player_rest_lookup.duplicated(
    player_key
).any()

research = safe_left_merge(
    research,
    player_rest_lookup,
    on=player_key,
    relationship="one_to_one",
    label="player rest merge",
)

# ============================================================
# 12B. Team rest and within-GW fixture order
# ============================================================

team_schedule_cols = [
    "season",
    "fixture_id",
    "team_id",
    "kickoff_time",
    "gameweek",
]

team_schedule = (
    research[team_schedule_cols]
    .drop_duplicates(
        ["season", "fixture_id", "team_id"]
    )
    .sort_values(
        [
            "season",
            "team_id",
            "kickoff_time",
            "fixture_id",
        ],
        kind="stable",
    )
    .copy()
)

team_schedule_key = [
    "season",
    "fixture_id",
    "team_id",
]

assert not team_schedule.duplicated(
    team_schedule_key
).any()

team_schedule["previous_team_fixture_time"] = (
    team_schedule
    .groupby(
        ["season", "team_id"],
        sort=False,
    )["kickoff_time"]
    .shift(1)
)

team_schedule["days_since_team_fixture"] = (
    (
        team_schedule["kickoff_time"]
        - team_schedule["previous_team_fixture_time"]
    )
    .dt.total_seconds()
    .div(86_400)
)

team_schedule["fixture_number_in_gameweek"] = (
    team_schedule
    .groupby(
        ["season", "team_id", "gameweek"],
        sort=False,
    )
    .cumcount()
    .add(1)
)

team_schedule["is_additional_gw_fixture"] = (
    team_schedule["fixture_number_in_gameweek"] > 1
)

team_schedule["short_rest_3d"] = (
    team_schedule["days_since_team_fixture"] <= 3
).astype("boolean")

team_schedule["short_rest_4d"] = (
    team_schedule["days_since_team_fixture"] <= 4
).astype("boolean")

cold_start_team = (
    team_schedule["days_since_team_fixture"].isna()
)

team_schedule.loc[
    cold_start_team,
    ["short_rest_3d", "short_rest_4d"],
] = pd.NA

team_schedule_lookup = team_schedule[
    team_schedule_key
    + [
        "days_since_team_fixture",
        "fixture_number_in_gameweek",
        "is_additional_gw_fixture",
        "short_rest_3d",
        "short_rest_4d",
    ]
].copy()

research = safe_left_merge(
    research,
    team_schedule_lookup,
    on=team_schedule_key,
    relationship="many_to_one",
    label="team schedule merge",
)

# ============================================================
# 12C. Current fixture context
# ============================================================

if "was_home" in research.columns:
    research["is_home"] = (
        research["was_home"]
        .map({True: 1, False: 0})
        .astype("Int64")
    )

print(
    research[
        [
            c for c in [
                "season",
                "fixture_id",
                "player_id",
                "team_id",
                "opponent_team_id",
                "days_since_player_fixture",
                "days_since_last_appearance",
                "days_since_team_fixture",
                "fixture_number_in_gameweek",
                "is_additional_gw_fixture",
                "short_rest_3d",
                "short_rest_4d",
                "is_home",
            ]
            if c in research.columns
        ]
    ].head(20)
)


     season  fixture_id  player_id  team_id  opponent_team_id  \
0   2016-17           8          6        1                 9   
1   2016-17          13          6        1                 8   
2   2016-17          28          6        1                18   
3   2016-17          31          6        1                13   
4   2016-17          43          6        1                 7   
5   2016-17          51          6        1                 4   
6   2016-17          61          6        1                 3   
7   2016-17          71          6        1                16   
8   2016-17          81          6        1                12   
9   2016-17          97          6        1                15   
10  2016-17         101          6        1                17   
11  2016-17         113          6        1                11   
12  2016-17         121          6        1                 2   
13  2016-17         140          6        1                20   
14  2016-17         141  

## 13. Cold-start policy

Do not remove players merely because they have fewer than two appearances.

Instead expose their history depth.

Candidate features:
- prior fixtures;
- prior minutes;
- low-history flag;
- position;
- team strength;
- price.

Later model experiments can determine whether specialized priors are needed.

In [19]:
cold_start_summary = (
    research["player_prior_fixtures"]
    .value_counts()
    .sort_index()
    .rename_axis("prior_fixtures")
    .to_frame("rows")
)

display(cold_start_summary.head(15))

,rows
prior_fixtures,
0,2786
1,2772
2,2764
3,2755
4,2746
5,2733
6,2728
7,2723
8,2715


## 14. Leakage assertions

These checks are more important than feature correlation.

A feature dataset should fail if:
- any feature is derived from the target row;
- chronological order is ambiguous;
- merge row counts change;
- target columns accidentally appear among predictor columns.

In [20]:
TARGET_FORBIDDEN_RAW = {
    "total_points",
    "upcoming_total_points",
    "target_points",
}

candidate_predictors = [
    c for c in research.columns
    if c not in {"target_points", "player_name", "kickoff_time"}
]

unexpected_target_names = TARGET_FORBIDDEN_RAW.intersection(candidate_predictors)
assert not unexpected_target_names, (
    f"Target leakage columns found: {unexpected_target_names}"
)

assert len(research) == len(df), (
    f"Feature table row count changed: canonical={len(df):,}, research={len(research):,}"
)

raw_outcome_cols = set(PLAYER_HISTORY_STATS)
unexpected_raw_outcomes = raw_outcome_cols.intersection(candidate_predictors)
assert not unexpected_raw_outcomes, (
    f"Raw same-fixture outcome predictors found: {sorted(unexpected_raw_outcomes)}"
)

print("Leakage-name and cardinality checks passed.")
print("Synthetic player/team value-level tests also passed earlier.")


Leakage-name and cardinality checks passed.
Synthetic player/team value-level tests also passed earlier.


## 15. Feature missingness is research evidence

Do not `dropna()` here.

Missing early-history features are expected and informative.

Quantify missingness so the model-experiment notebook can choose:
- explicit imputation;
- native missing-value handling;
- missingness indicators;
- cold-start priors.

In [21]:
feature_missingness = (
    research.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)

display(feature_missingness.head(50))

,missing_pct
influence_per90_season_to_date,38.411546
bps_per90_season_to_date,38.411546
bonus_per90_season_to_date,38.411546
assists_per90_season_to_date,38.411546
goals_scored_per90_season_to_date,38.411546
creativity_per90_season_to_date,38.411546
ict_index_per90_season_to_date,38.411546
threat_per90_season_to_date,38.411546
clean_sheets_per90_season_to_date,38.411546
total_points_per90_season_to_date,38.411546


## 16. Sanity plots — features versus time/history depth

We are not optimizing a model here.

The goal is to detect absurd feature behaviour:
- season-to-date totals decreasing;
- rolling windows including target outcomes;
- exploding per-90 rates;
- current fixture leaking into history.

In [22]:
numeric_candidates = [
    c for c in [
        "player_prior_fixtures",
        "minutes_season_to_date",
        "total_points_season_to_date",
        "total_points_last_3_fixtures",
        "total_points_last_6_fixtures",
        "total_points_per90_season_to_date",
    ]
    if c in research.columns
]

for col in numeric_candidates:
    fig, ax = plt.subplots(figsize=(8, 4))
    research[col].dropna().hist(bins=50, ax=ax)
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("Rows")
    plt.show()

/tmp/ipykernel_47853/1110410044.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 17. Feature-family and point-in-time availability contract

A feature is not approved merely because it exists in the training table.

Every predictor must document:
- feature family;
- source/time semantics;
- the rule that makes it reproducible before future kickoff;
- whether it is approved, conditional, or still requires source validation.

In particular, `value` remains **unapproved until its historical timestamp semantics are verified**.


In [23]:
def feature_family(name):
    if name.startswith("team_"):
        return "team_history"
    if name.startswith("opp_"):
        return "opponent_history"
    if "minutes" in name or "60plus" in name:
        return "minutes_availability"
    if "_per90_" in name:
        return "player_rate"
    if "_last_" in name:
        return "player_recent"
    if name.endswith("_season_to_date"):
        return "player_season_to_date"
    if name in {
        "is_home", "was_home", "fixture_number_in_gameweek", "is_additional_gw_fixture",
        "days_since_player_fixture", "days_since_last_appearance",
        "days_since_team_fixture", "short_rest_3d", "short_rest_4d",
    }:
        return "fixture_context"
    if name in {"position", "value"}:
        return "player_context"
    return "identifier_or_other"


def availability_contract(name):
    if name == "value":
        return (
            "requires_source_validation",
            "must be the player's value known before target kickoff/deadline",
            "Verify historical source timestamp semantics before modelling.",
        )

    if name in {"fixture_number_in_gameweek", "is_additional_gw_fixture"}:
        return (
            "conditional",
            "future fixture schedule and ordering must be known as of prediction time",
            "Rescheduling can change the ordering.",
        )

    if name in {"is_home", "was_home", "position"}:
        return (
            "approved",
            "known from pre-kickoff fixture/player context",
            "",
        )

    if name.startswith(("team_", "opp_")):
        return (
            "approved",
            "derived only from fixtures before target kickoff",
            "",
        )

    if (
        "_last_" in name
        or name.endswith("_season_to_date")
        or "_per90_" in name
        or name in {
            "player_prior_fixtures",
            "low_history_minutes_flag",
            "days_since_player_fixture",
            "days_since_last_appearance",
            "days_since_team_fixture",
            "short_rest_3d",
            "short_rest_4d",
        }
    ):
        return (
            "approved",
            "derived only from prior fixtures after shift(1)",
            "",
        )

    return (
        "conditional",
        "availability rule not yet explicitly classified",
        "Review before allowing into model experiments.",
    )


identifier_cols = {
    "season", "player_id", "player_name", "fixture_id", "gameweek",
    "kickoff_time", "team_id", "team_name", "opponent_team_id",
    "opponent_team_name", "target_points",
}

feature_names = [c for c in research.columns if c not in identifier_cols]
contract = [availability_contract(c) for c in feature_names]

feature_dictionary = pd.DataFrame({
    "feature": feature_names,
    "family": [feature_family(c) for c in feature_names],
    "availability_status": [x[0] for x in contract],
    "point_in_time_rule": [x[1] for x in contract],
    "notes": [x[2] for x in contract],
})

display(feature_dictionary)
display(feature_dictionary["availability_status"].value_counts().rename("features"))


,feature,family,availability_status,point_in_time_rule,notes
0,team_name_x,team_history,approved,derived only from fixtures before target kickoff,
1,position,player_context,approved,known from pre-kickoff fixture/player context,
2,was_home,fixture_context,approved,known from pre-kickoff fixture/player context,
3,value,player_context,requires_source_validation,must be the player's value known before target...,Verify historical source timestamp semantics b...
4,player_prior_fixtures,identifier_or_other,approved,derived only from prior fixtures after shift(1),
...,...,...,...,...,...
150,fixture_number_in_gameweek,fixture_context,conditional,future fixture schedule and ordering must be k...,Rescheduling can change the ordering.
151,is_additional_gw_fixture,fixture_context,conditional,future fixture schedule and ordering must be k...,Rescheduling can change the ordering.
152,short_rest_3d,fixture_context,approved,derived only from prior fixtures after shift(1),
153,short_rest_4d,fixture_context,approved,derived only from prior fixtures after shift(1),


availability_status
approved                      152
conditional                     2
requires_source_validation      1
Name: features, dtype: int64

## 18. Research checks before model experiments

Before export, resolve the following:

### Temporal correctness
- Do all rolling/cumulative player and team features exclude the target fixture?
- Are DGWs and postponements ordered by actual kickoff time?
- Do the synthetic leakage tests still pass after any feature-builder change?

### Context availability
- Verify that historical `value` is the price known before the target prediction time.
- Treat `fixture_number_in_player_gw` as conditional on the schedule known at prediction time.
- Confirm any future FPL snapshot field has an explicit source timestamp.

### Modelling scope
- Keep zero-minute target rows for the initial single-stage expected-points baseline.
- Compare a two-stage appearance/conditional-points formulation later rather than filtering those rows now.

### Future reproducibility
For every candidate predictor ask:

> Can this exact value be generated for a fixture that has not happened yet, using only information available at prediction time?

If not, it must not enter model experiments.


## 19. Optional next feature families

Add only after the baseline feature table is proven correct:

### High priority
- expected minutes / chance of playing;
- true starting-XI / substitution indicators (if timestamp-safe);
- xG / xA / xGI;
- shots / shots in box;
- key passes / big chances;
- team xG;
- opponent xGA;
- home/away-specific team strength;
- price and price changes;
- rest days.

### Medium priority
- set-piece role;
- penalties;
- corners/free kicks;
- fixture congestion;
- promoted-team indicator;
- manager/team regime changes.

### Avoid initially
- high-cardinality player-name encoding;
- opaque manually constructed "form" scores;
- features derived after deadline/kickoff;
- league ranks calculated using matches later in the same GW.


## 20. Save research feature table

Do not perform feature selection here using the full dataset.

Feature selection belongs **inside chronological training folds** in `03_model_experiments.ipynb`.

Likewise:
- no global scaler fitting;
- no random train/test split;
- no model-based elimination.

In [24]:
model_key = [c for c in ["season", "fixture_id", "player_id"] if c in research.columns]

if len(model_key) == 3:
    assert not research.duplicated(model_key).any(), (
        "Final fixture feature table violates player-fixture uniqueness."
    )

assert len(research) == len(df)
assert research["target_points"].notna().all()

modelling_blockers = feature_dictionary[
    feature_dictionary["availability_status"].eq("requires_source_validation")
]

print("Rows:", f"{len(research):,}")
print("Candidate features:", len(feature_names))
print("Target missing:", int(research["target_points"].isna().sum()))

if len(modelling_blockers):
    print("\nFEATURE TABLE NOT YET APPROVED FOR MODEL EXPERIMENTS")
    display(
        modelling_blockers[
            ["feature", "availability_status", "point_in_time_rule", "notes"]
        ]
    )
else:
    print("\nAll source-validation blockers resolved.")

# Enable only after source-validation blockers are resolved.
# research.to_parquet(FEATURE_OUTPUT, index=False)
# feature_dictionary.to_csv(FEATURE_DICTIONARY_OUTPUT, index=False)


Rows: 96,169
Candidate features: 155
Target missing: 0

FEATURE TABLE NOT YET APPROVED FOR MODEL EXPERIMENTS


,feature,availability_status,point_in_time_rule,notes
3,value,requires_source_validation,must be the player's value known before target...,Verify historical source timestamp semantics b...


# Exit criteria for Notebook 02

Do not move to model experimentation until:

- [ ] one row = one player + one target fixture;
- [ ] target is current-fixture points, not a shifted "next row";
- [ ] player and team history builders pass synthetic value-level leakage tests;
- [ ] all historical features use only fixtures before target kickoff;
- [ ] DGWs remain separate target fixtures;
- [ ] chronology is based on `kickoff_time`, not plain gameweek number;
- [ ] gameweek-window aggregates remain disabled unless implemented with target-time as-of logic;
- [ ] team and opponent lookups are unique before merging;
- [ ] every merge preserves row count exactly;
- [ ] cold-start and zero-minute target rows remain present;
- [ ] no blanket `dropna()` is used;
- [ ] `value` timestamp semantics are verified before it is approved as a predictor;
- [ ] schedule-derived features are marked conditional where appropriate;
- [ ] every candidate predictor has an explicit point-in-time availability rule;
- [ ] raw target/same-fixture outcome columns cannot enter predictors;
- [ ] no scaler, selector, or model is fitted globally;
- [ ] the feature table can be regenerated deterministically.
